# 04-01 — Chunking (real DI + Vision → chunks → embeddings)

This notebook plugs into the same flow as `02_doc_intelligence.ipynb`
and `03_openai_vision.ipynb`, but stops one step further down the
pipeline:

    Blob → Document Intelligence → PyMuPDF + GPT-4o vision →
    **chunking** → **embedding**

It does **not** index into AI Search — that is `04_ai_search.ipynb`.

Place a PDF at `notebooks/sample.pdf` (or update `PDF_PATH` below) before running.

## 1. Setup — services + upload PDF


In [1]:
import os, sys, pathlib
sys.path.insert(0, os.path.abspath('..'))

from src.blob_client import BlobService
from src.doc_intelligence import DocIntelService
from src.openai_client import OpenAIService
from src.chunking import (
    CHUNK_OVERLAP,
    CHUNK_TOKENS,
    assemble_chunks,
    build_image_chunks,
)

PDF_PATH = pathlib.Path('Multi_Agent_Research_System_Architecture.pdf')
assert PDF_PATH.exists(), 'Drop a sample.pdf next to this notebook'

DOC_ID = 'docmind-chunk-test'
BLOB_NAME = f'{DOC_ID}/sample.pdf'

blob = BlobService()
blob.ensure_container()
pdf_bytes = PDF_PATH.read_bytes()
blob.upload(BLOB_NAME, pdf_bytes, content_type='application/pdf')
url = blob.get_url(BLOB_NAME)
print('PDF URL:', url[:80], '…')
print(f'max_chars={CHUNK_TOKENS * 4}, overlap_chars={CHUNK_OVERLAP * 4}')


INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'b24d77f3-4a5c-11f1-96aa-bcd22c162bea'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'f1e0d7fc-901e-0031-4b69-deb4ee000000'
    'x-ms-client-request-id': 'b24d77f3-4a5c-11f1-96aa-bcd22c162bea'
    'x-ms-version': 'REDAC

PDF URL: https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/sa …
max_chars=2000, overlap_chars=320


## 2. Document Intelligence — text + tables + figure regions

Same call as notebook 02. Returns per-page text, tables (as markdown),
and DI-detected figure regions.


In [2]:
doc_intel = DocIntelService()
result = doc_intel.extract_pdf(url)

text_pages = result['text_chunks']
tables = result['tables']
figures = result.get('figures', [])

print(f"pages       : {result['pages']}")
print(f"text pages  : {len(text_pages)}")
print(f"tables      : {len(tables)}")
print(f"DI figures  : {len(figures)}")
print('--- first 400 chars of page 1 ---')
print(text_pages[0]['content'][:400] if text_pages else '(empty)')


INFO: Document Intelligence analysing https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/sa
INFO: Request URL: 'https://azdocumentai.cognitiveservices.azure.com//documentintelligence/documentModels/prebuilt-layout:analyze?api-version=2024-11-30&outputContentFormat=REDACTED'
Request method: 'POST'
Request headers:
    'content-type': 'application/json'
    'Content-Length': '243'
    'Accept': 'application/json'
    'x-ms-client-request-id': 'bc5653a3-4a5c-11f1-a707-bcd22c162bea'
    'User-Agent': 'azsdk-python-ai-documentintelligence/1.0.2 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
A body is sent with the request
INFO: Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'REDACTED'
    'x-envoy-upstream-service-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'x-content-type-options': 'REDACTED'
    'x-ms-region': 'REDACTED'
   

pages       : 27
text pages  : 26
tables      : 24
DI figures  : 1
--- first 400 chars of page 1 ---
Multi-Agent Research System
Architecture Design & Technical Documentation
Author
Kumar Nishant - Senior Al Engineer
Team
MLEDSI
Manager
Krutika Kansara
Date
April 20, 2026
Version
1.2
Status
Draft - For Review


## 3. PyMuPDF + GPT-4o vision — image extraction & description

Same call as notebook 03. Each image is uploaded to Blob and described
by GPT-4o vision; the description is what we'll embed.


In [5]:
openai = OpenAIService()
images = doc_intel.extract_images(
    pdf_bytes,
    doc_id=DOC_ID,
    blob=blob,
    openai=openai,
    figures=figures,
)
print(f'extracted {len(images)} images')
for img in images[:5]:
    print(f"- page {img['page']}  source={img.get('source')} description={(img.get('description') or '')} caption={(img.get('caption') or '')[:60]!r}")


INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/figures/page15_fig0_0.png?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'Content-Length': '188895'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-blob-content-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '1c203f3f-4a5d-11f1-aeac-bcd22c162bea'
A body is sent with the request
INFO: Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Thu, 07 May 2026 21:38:40 GMT'
    'ETag': '"0x8DEAC81011497CB"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'f1e2f566-901e-0031-6069-deb4ee000000'

extracted 1 images
- page 15  source=figure description=This image is a flowchart depicting a process for handling a user query through a series of steps involving task management and research. Here is a detailed description of the components and their relationships:

1. **User Query**: The process begins with a user query.

2. **Planner Node**: The query is sent to the Planner Node.

3. **Analyze Intent + Define Goal**: The intent of the query is analyzed, and a goal is defined.

4. **Research Intent Classification**: The query is classified into categories such as blog, comparative, deep research, or summary.

5. **Structured Plan Creation**: A structured plan is created based on the classification.

6. **Generate Todos with status=pending**: Todos are generated with a pending status.

7. **Task Manager**: Manages the tasks and feeds into the Todo Selector.

8. **Todo Selector**: Selects pending tasks for processing.

9. **Delegation Layer**: Distributes tasks to sub-agents.

10. **S

## 4. Chunking — turn raw outputs into `ChunkRecord`s

`build_image_chunks` reshapes the vision-described images (no I/O),
then `assemble_chunks` does the sliding-window text chunking and
concatenates text + table + image chunks.


In [6]:
from collections import Counter

image_chunks = build_image_chunks(images, DOC_ID)

all_chunks = assemble_chunks(
    doc_id=DOC_ID,
    text_pages=text_pages,
    tables=tables,
    image_chunks=image_chunks,
)

counts = Counter(c.type for c in all_chunks)
print(f'total chunks : {len(all_chunks)}')
for t in ('text', 'table', 'image'):
    print(f'  {t:5s}      : {counts.get(t, 0)}')


total chunks : 56
  text       : 31
  table      : 24
  image      : 1


### Peek at one chunk of each type


In [7]:
for t in ('text', 'table', 'image'):
    sample = next((c for c in all_chunks if c.type == t), None)
    if not sample:
        continue
    print(f"--- {t.upper()} (page {sample.page}, {len(sample.content)} chars) ---")
    print(sample.content[:400])
    if sample.image_url:
        print('image_url:', sample.image_url[:80], '…')
    print()


--- TEXT (page 1, 209 chars) ---
Multi-Agent Research System
Architecture Design & Technical Documentation
Author
Kumar Nishant - Senior Al Engineer
Team
MLEDSI
Manager
Krutika Kansara
Date
April 20, 2026
Version
1.2
Status
Draft - For Review

--- TABLE (page 1, 185 chars) ---
| Author | Kumar Nishant - Senior Al Engineer |
| --- | --- |
| Team | MLEDSI |
| Manager | Krutika Kansara |
| Date | April 20, 2026 |
| Version | 1.2 |
| Status | Draft - For Review |

--- IMAGE (page 15, 1754 chars) ---
[Figure on page 15]: This image is a flowchart depicting a process for handling a user query through a series of steps involving task management and research. Here is a detailed description of the components and their relationships:

1. **User Query**: The process begins with a user query.

2. **Planner Node**: The query is sent to the Planner Node.

3. **Analyze Intent + Define Goal**: The intent
image_url: https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/fi …



## 5. Sanity checks

- text chunks respect `max_chars`
- adjacent text chunks on the same page overlap by `overlap_chars`
- image chunks carry `image_url` + page header


In [8]:
from itertools import groupby

MAX_CHARS = CHUNK_TOKENS * 4
OVERLAP_CHARS = CHUNK_OVERLAP * 4

text_chunks = [c for c in all_chunks if c.type == 'text']
oversize = [c for c in text_chunks if len(c.content) > MAX_CHARS]
print(f'oversize text chunks: {len(oversize)} (expect 0)')

# Group consecutive text chunks per page and verify overlap
bad_overlaps = 0
for page, group in groupby(text_chunks, key=lambda c: c.page):
    g = list(group)
    for a, b in zip(g, g[1:]):
        if a.content[-OVERLAP_CHARS:] != b.content[:OVERLAP_CHARS]:
            bad_overlaps += 1
print(f'bad overlap pairs   : {bad_overlaps} (expect 0)')

img_chunks = [c for c in all_chunks if c.type == 'image']
missing_url = [c for c in img_chunks if not c.image_url]
print(f'image chunks missing url: {len(missing_url)} (expect 0)')
if img_chunks:
    print('sample image header :', img_chunks[0].content.split(']')[0] + ']')


oversize text chunks: 0 (expect 0)
bad overlap pairs   : 0 (expect 0)
image chunks missing url: 0 (expect 0)
sample image header : [Figure on page 15]


## 6. Embedding — `text-embedding-ada-002`

Mirrors the embed step inside `IngestionPipeline.process_pdf`: batches
of 16, attached to each `ChunkRecord.embedding`.


In [ ]:
BATCH = 16
texts = [c.content for c in all_chunks]
vectors: list[list[float]] = []
for i in range(0, len(texts), BATCH):
    vectors.extend(openai.embed(texts[i:i + BATCH]))

for c, v in zip(all_chunks, vectors):
    c.embedding = v

dims = {len(v) for v in vectors}
print(f'embedded {len(vectors)} chunks; embedding dim(s): {dims}')
assert len(vectors) == len(all_chunks)
assert all(c.embedding for c in all_chunks)
print('✓ every chunk has an embedding')


## 7. Quick semantic-search sanity (optional)

Cosine-similarity over the in-memory chunks — a fast way to eyeball
whether chunking + embedding produced sensible vectors before pushing
to AI Search.


In [ ]:
import math

def cosine(a, b):
    s = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return s / (na * nb) if na and nb else 0.0

QUERY = 'multi-agent system architecture'
qvec = openai.embed([QUERY])[0]

scored = sorted(
    ((cosine(qvec, c.embedding), c) for c in all_chunks),
    key=lambda x: x[0],
    reverse=True,
)[:5]

for score, c in scored:
    print(f'{score:.3f}  [{c.type:5s} p{c.page}]  {c.content[:120].strip()}…')
